# Aquaculture Model Analysis

This notebook demonstrates how to analyze a trained model from the aquaculture competition framework using actual competition data.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import os
import random
from pathlib import Path
import sys

# Add the parent directory to the system path to import local modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import our custom modules
from aquaculture.feature_engineering import AquacultureFeatureEngineer
from src.inference import load_inference_pipeline
from src.plotting import (
    plot_feature_importance, plot_roc_curve, plot_precision_recall_curve,
    plot_confusion_matrix, plot_calibration_curve
)
from src.metrics import calculate_metrics, competition_score, calculate_roc_curve, calculate_precision_recall_curve
from sklearn.metrics import confusion_matrix
from sklearn.calibration import calibration_curve

# For reproducibility
np.random.seed(42)
random.seed(42)

# Set up paths
DATA_DIR = Path('../data')
EXPERIMENTS_DIR = Path('../experiments')

# Try to find the most recent experiment directory
if EXPERIMENTS_DIR.exists():
    experiment_dirs = [d for d in EXPERIMENTS_DIR.iterdir() if d.is_dir()]
    if experiment_dirs:
        # Sort by modification time (newest first)
        experiment_dirs.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_experiment = experiment_dirs[0]
        print(f"Found experiment: {latest_experiment.name}")
    else:
        print("No experiment directories found!")
        import sys
        sys.exit(1)
else:
    print("Experiments directory not found!")
    import sys
    sys.exit(1)

# Load the inference pipeline
print("Loading inference pipeline...")
try:
    pipeline = load_inference_pipeline(str(latest_experiment))
    print("✓ Inference pipeline loaded successfully")
    print(f"Model type: {type(pipeline.model).__name__}")
    if pipeline.feature_names:
        print(f"Number of features: {len(pipeline.feature_names)}")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please check that the experiment directory exists and contains a trained model")
    import sys
    sys.exit(1)

## 2. Load Features and Generate Predictions

Load the engineered features and true labels, then generate predictions using the loaded model pipeline.

In [2]:
# Load features and labels
print("Loading features and labels...")
features_df = pd.read_parquet(latest_experiment / "features.parquet")
labels_df = pd.read_csv(DATA_DIR / "Train.csv")
y = labels_df["label"].values
print(f"Features shape: {features_df.shape}")
print(f"Labels shape: {y.shape}")

# Prepare data for analysis
print("Preparing data for analysis...")
# The features DataFrame already contains only the engineered features
X = features_df.values

# Make predictions
print("Generating predictions...")
predictions = pipeline.predict(X)
probabilities = pipeline.predict_proba(X)

Loading features and labels...
Features shape: (1821, 431)
Labels shape: (1821,)
Preparing data for analysis...
Generating predictions...


/Users/fortin/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/fortin/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 3. Evaluate Model Performance

Calculate various metrics to evaluate the model's performance.

In [3]:
# Calculate metrics for single target
print(f"\n=== Target Evaluation ===")
target_pred = predictions
target_prob = probabilities
target_true = y

# Calculate various metrics
metrics = calculate_metrics(target_true, target_prob)

# Print key metrics
print(f"Accuracy:  {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall:    {metrics['recall']:.4f}")
print(f"F1-Score:  {metrics['f1']:.4f}")
print(f"ROC AUC:   {metrics['roc_auc']:.4f}")
print(f"PR AUC:    {metrics['pr_auc']:.4f}")

# Calculate competition score (for single target, this is just the standard competition score)
competition_score_value = competition_score(target_true, target_prob)
print(f"\nCompetition Score: {competition_score_value:.4f}")


=== Target Evaluation ===
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
ROC AUC:   1.0000
PR AUC:    1.0000

Competition Score: 1.0000


## 4. Generate Visualizations

Create diagnostic plots to understand model performance.

In [ ]:
# Generate plots for single target
print(f"\nGenerating plots for target...")
target_pred = predictions
target_prob = probabilities
target_true = y

# Create a directory for plots
PLOTS_DIR = EXPERIMENTS_DIR / latest_experiment.name / 'plots'
PLOTS_DIR.mkdir(exist_ok=True)

# ROC Curve
fpr, tpr, _ = calculate_roc_curve(target_true, target_prob)
roc_auc = metrics['roc_auc']
plot_roc_curve(fpr, tpr, roc_auc,
               title=f'ROC Curve',
               save_path=PLOTS_DIR / f'roc_curve.png')

# Precision-Recall Curve
precision, recall, _ = calculate_precision_recall_curve(target_true, target_prob)
pr_auc = metrics['pr_auc']
plot_precision_recall_curve(precision, recall, pr_auc,
                            title=f'Precision-Recall Curve',
                            save_path=PLOTS_DIR / f'pr_curve.png')

# Confusion Matrix
cm = confusion_matrix(target_true, target_pred)
plot_confusion_matrix(cm,
                      title=f'Confusion Matrix',
                      save_path=PLOTS_DIR / f'confusion_matrix.png')

# Calibration Curve
prob_true, prob_pred = calibration_curve(target_true, target_prob, n_bins=10)
plot_calibration_curve(prob_true, prob_pred,
                       title=f'Calibration Curve',
                       save_path=PLOTS_DIR / f'calibration_curve.png')

print("\nAll plots generated successfully!")